In [1]:
# Check environment
import subprocess, os

has_gpu = subprocess.run('nvidia-smi', shell=True, capture_output=True).returncode == 0
has_nvcc = subprocess.run('nvcc --version', shell=True, capture_output=True).returncode == 0

print('GPU available :', has_gpu)
print('nvcc available:', has_nvcc)
if has_gpu:
    !nvidia-smi
    !nvcc --version
else:
    print('No GPU — will use CPU-only build (Cell 4)')
    !g++ --version | head -1

GPU available : False
nvcc available: False
No GPU — will use CPU-only build (Cell 4)
g++ (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0


In [2]:
# Install dependencies
!apt-get install -y libtiff-dev libomp-dev liblz4-dev 2>&1 | tail -5
!dpkg -l libtiff-dev liblz4-dev | grep '^ii'


/sbin/ldconfig.real: /usr/local/lib/libhwloc.so.15 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc.so.2 is not a symbolic link

ii  liblz4-dev:amd64  1.9.3-2build2     amd64        Fast LZ compression algorithm library - development files
ii  libtiff-dev:amd64 4.3.0-6ubuntu0.13 amd64        Tag Image File Format library (TIFF), development files


In [3]:
#upload files

from google.colab import files
import os

os.makedirs('/content/src', exist_ok=True)
uploaded = files.upload()

for fname, data in uploaded.items():
    dest = f'/content/src/{fname}'
    with open(dest, 'wb') as f:
        f.write(data)
    print(f'  Saved {fname} ({len(data)//1024} KB)')

print('\nAll files in /content/src:')
!ls -lh /content/src/

Saving AsyncGPUWorker.cpp to AsyncGPUWorker.cpp
Saving AsyncGPUWorker.h to AsyncGPUWorker.h
Saving BoundedTileQueue.h to BoundedTileQueue.h
Saving CPUWorker.h to CPUWorker.h
Saving CUDAKernels.cu to CUDAKernels.cu
Saving CUDAKernels.cuh to CUDAKernels.cuh
Saving CUDAKernels.h to CUDAKernels.h
Saving CUDAKernelsStub.cpp to CUDAKernelsStub.cpp
Saving CUDAStreamManager.cpp to CUDAStreamManager.cpp
Saving CUDAStreamManager.h to CUDAStreamManager.h
Saving GPUWorker.h to GPUWorker.h
Saving input.tiff to input.tiff
Saving IWorker.h to IWorker.h
Saving main.cpp to main.cpp
Saving OptimizedCPUWorker.cpp to OptimizedCPUWorker.cpp
Saving OptimizedCPUWorker.h to OptimizedCPUWorker.h
Saving OutputWriter.cpp to OutputWriter.cpp
Saving OutputWriter.h to OutputWriter.h
Saving Scheduler.h to Scheduler.h
Saving Tile.h to Tile.h
Saving TileOptimizer.cpp to TileOptimizer.cpp
Saving TileOptimizer.h to TileOptimizer.h
Saving TileReader.cpp to TileReader.cpp
Saving TileReader.h to TileReader.h
Saving transfo

**Compile on GPU**

In [6]:
import subprocess
if subprocess.run('which nvcc', shell=True, capture_output=True).returncode != 0:
    print('nvcc not found — skipping GPU build. Use CPU binary from Cell 5.')
else:
    GPU_CMD = """
    cd /content/src && nvcc \\
      -std=c++17 \\
      -O2 \\
      -DUSE_CUDA \\
      -arch=sm_75 \\
      -Xcompiler -fopenmp \\
      -I. \\
      main.cpp \\
      TileReader.cpp \\
      transform.cpp \\
      OptimizedCPUWorker.cpp \\
      AsyncGPUWorker.cpp \\
      CUDAStreamManager.cpp \\
      VramManager.cpp \\
      TileOptimizer.cpp \\
      OutputWriter.cpp \\
      CUDAKernels.cu \\
      -L/usr/lib/x86_64-linux-gnu -ltiff -llz4 \\
      -lpthread \\
      -lgomp \\
      -o /content/gigapixel_pipeline \\
      2>&1
    """
    !{GPU_CMD}
    !ls -lh /content/gigapixel_pipeline 2>/dev/null && echo 'GPU build OK' || echo 'GPU build FAILED'

/bin/bash: line 2: cd: too many arguments
Build FAILED — see errors above


**Compile on CPU**

In [4]:
# Write cuda_runtime.h stub (CPU-only build only)
#
# Tile.h, VramManager.h, CUDAStreamManager.h all include <cuda_runtime.h>.
# On a CPU-only runtime that header does not exist, so the build fails
# immediately. This cell writes a minimal stub that satisfies every
# declaration those files need, without actually calling any CUDA API.
#
# The stub is written INTO /content/src/ so -I. in the compile command
# finds it before any system include path.
#
# Safe to run even if CUDA IS available — the real cuda_runtime.h from
# /usr/local/cuda/include will be found first via nvcc's default paths,
# and the stub in /content/src/ is only used by g++.

stub = r"""
#pragma once
// cpu_stub/cuda_runtime.h
// Minimal stub so CUDA-aware headers compile with plain g++ (no CUDA installed).
#include <cstdint>
#include <cstddef>
#include <cstdio>

typedef void*  cudaStream_t;
typedef int    cudaError_t;
typedef void*  cudaDeviceProp;

static const cudaError_t cudaSuccess = 0;

inline const char* cudaGetErrorString(cudaError_t) { return "(cuda stub)"; }

// Memory
inline cudaError_t cudaMalloc(void**, size_t)                              { return 1; }
inline cudaError_t cudaFree(void*)                                         { return 1; }
inline cudaError_t cudaMallocHost(void** ptr, size_t size)                 {
    *ptr = ::operator new(size); return *ptr ? 0 : 1;
}
inline cudaError_t cudaFreeHost(void* ptr)                                 {
    ::operator delete(ptr); return 0;
}
inline cudaError_t cudaMemcpyAsync(void*,const void*,size_t,int,cudaStream_t) { return 1; }

// Streams
inline cudaError_t cudaStreamCreateWithFlags(cudaStream_t*, int)           { return 1; }
inline cudaError_t cudaStreamDestroy(cudaStream_t)                         { return 0; }
inline cudaError_t cudaStreamSynchronize(cudaStream_t)                     { return 0; }
inline cudaError_t cudaDeviceSynchronize()                                 { return 0; }

// Enums used in the codebase
enum {
    cudaStreamNonBlocking   = 1,
    cudaMemcpyHostToDevice  = 1,
    cudaMemcpyDeviceToHost  = 2,
    cudaMemcpyDeviceToDevice= 3
};

// uchar3 used in CUDAKernels.h / .cuh
struct uchar3 { unsigned char x, y, z; };
"""

with open('/content/src/cuda_runtime.h', 'w') as f:
    f.write(stub)
print('Written: /content/src/cuda_runtime.h')

# Also write stub implementations for the three .cpp files that call
# real CUDA APIs.  On CPU these classes are constructed but never
# reach the CUDA call sites (Scheduler routes everything to CPU),
# so empty / error-returning stubs are safe.

vram_stub = r"""
// VramManager_cpu.cpp  — CPU-only stub
// Replaces VramManager.cpp when building without CUDA.
// acquireBuffer() allocates normal heap memory; releaseBuffer() frees it.
// The Scheduler never calls the GPU path on CPU-only builds, so these
// buffers are acquired and immediately released without use.
#include "VramManager.h"
#include <new>
#include <stdexcept>

VramManager::VramManager(int numBuffers, size_t bufferSizeBytes)
    : bufferSize(bufferSizeBytes)
{
    for (int i = 0; i < numBuffers; ++i) {
        uint8_t* p = new (std::nothrow) uint8_t[bufferSizeBytes];
        if (!p) throw std::runtime_error("VramManager stub: heap alloc failed");
        freeBuffers.push(p);
        allAllocations.push_back(p);
    }
}

VramManager::~VramManager() {
    for (uint8_t* p : allAllocations) delete[] p;
    allAllocations.clear();
    while (!freeBuffers.empty()) freeBuffers.pop();
}

uint8_t* VramManager::acquireBuffer() {
    std::unique_lock<std::mutex> lock(mtx);
    cv.wait(lock, [this]{ return !freeBuffers.empty(); });
    uint8_t* p = freeBuffers.front(); freeBuffers.pop();
    return p;
}

void VramManager::releaseBuffer(uint8_t* ptr) {
    if (!ptr) return;
    { std::lock_guard<std::mutex> lock(mtx); freeBuffers.push(ptr); }
    cv.notify_one();
}
"""

stream_stub = r"""
// CUDAStreamManager_cpu.cpp  — CPU-only stub
#include "CUDAStreamManager.h"
#include <iostream>

CUDAStreamManager::CUDAStreamManager(int n) : numStreams(n), profilingEnabled(false) {
    streams.resize(n, nullptr);
    streamAvailable.resize(n, true);
    std::cout << "[CUDAStreamManager] CPU stub: " << n << " dummy streams." << std::endl;
}
CUDAStreamManager::~CUDAStreamManager() {}

int  CUDAStreamManager::acquireStream() {
    std::lock_guard<std::mutex> lock(poolMutex);
    for (int i = 0; i < numStreams; ++i)
        if (streamAvailable[i]) { streamAvailable[i] = false; return i; }
    return 0;
}
void CUDAStreamManager::launchH2DTransfer(int,void*,const void*,size_t){}
void CUDAStreamManager::launchD2HTransfer(int,void*,const void*,size_t){}
void CUDAStreamManager::synchronizeStream(int id) {
    std::lock_guard<std::mutex> lock(poolMutex);
    if (id >= 0 && id < numStreams) streamAvailable[id] = true;
}
void CUDAStreamManager::synchronizeAllStreams() {
    std::lock_guard<std::mutex> lock(poolMutex);
    std::fill(streamAvailable.begin(), streamAvailable.end(), true);
}
cudaStream_t CUDAStreamManager::getRawCudaStream(int) const { return nullptr; }
int          CUDAStreamManager::getNumStreams()           const { return numStreams; }
void         CUDAStreamManager::enableProfiling(bool e)         { profilingEnabled = e; }
void         CUDAStreamManager::checkCudaError(cudaError_t,const char*){}
"""

async_stub = r"""
// AsyncGPUWorker_cpu.cpp  — CPU-only stub
// GPU path is never reached on CPU builds (Scheduler always picks CPU),
// but the object is still constructed, so we need a valid implementation.
#include "AsyncGPUWorker.h"
#include "Transform.h"
#include <iostream>
#include <stdexcept>

AsyncGPUWorker::AsyncGPUWorker() : currentGPU(0) {
    streamManager = std::make_unique<CUDAStreamManager>(3);
    gpuInfo = {0, 0, 0, 0, 0, 0};
    std::cout << "[AsyncGPUWorker] CPU stub — GPU calls are no-ops." << std::endl;
}
AsyncGPUWorker::~AsyncGPUWorker() { try { waitForCompletion(); } catch(...){} }
void AsyncGPUWorker::initializeGPU() {}
void AsyncGPUWorker::setGPUDevice(int id) { currentGPU = id; }
void AsyncGPUWorker::execute(Tile& tile, uint8_t*) {
    std::cout << "[AsyncGPUWorker] CPU stub: falling back for tile ("
              << tile.x << "," << tile.y << ")" << std::endl;
    processTile(tile);
}
void AsyncGPUWorker::executeGPUKernel(Tile&,uint8_t*,int){}
void AsyncGPUWorker::waitForCompletion() { if(streamManager) streamManager->synchronizeAllStreams(); }
void AsyncGPUWorker::recordPerformance(){}
"""

with open('/content/src/VramManager_cpu.cpp',       'w') as f: f.write(vram_stub)
with open('/content/src/CUDAStreamManager_cpu.cpp', 'w') as f: f.write(stream_stub)
with open('/content/src/AsyncGPUWorker_cpu.cpp',    'w') as f: f.write(async_stub)
print('Written: VramManager_cpu.cpp')
print('Written: CUDAStreamManager_cpu.cpp')
print('Written: AsyncGPUWorker_cpu.cpp')

Written: /content/src/cuda_runtime.h
Written: VramManager_cpu.cpp
Written: CUDAStreamManager_cpu.cpp
Written: AsyncGPUWorker_cpu.cpp


In [14]:
#cpu compile
CPU_CMD = """
cd /content/src && g++ \\
  -std=c++17 \\
  -O2 \\
  -fopenmp \\
  -I. \\
  main.cpp \\
  TileReader.cpp \\
  transform.cpp \\
  OptimizedCPUWorker.cpp \\
  TileOptimizer.cpp \\
  OutputWriter.cpp \\
  CUDAKernelsStub.cpp \\
  VramManager_cpu.cpp \\
  CUDAStreamManager_cpu.cpp \\
  AsyncGPUWorker_cpu.cpp \\
  -L/usr/lib/x86_64-linux-gnu -ltiff -llz4 \\
  -lpthread \\
  -lgomp \\
  -o /content/gigapixel_cpu \\
  2>&1
"""
!{CPU_CMD}
!ls -lh /content/gigapixel_cpu 2>/dev/null && echo 'CPU build OK' || echo 'CPU build FAILED'

-rwxr-xr-x 1 root root 106K May  4 18:36 /content/gigapixel_cpu
CPU build OK


In [16]:
# Run
import time, subprocess, os

# Pick whichever binary compiled successfully
binary = '/content/gigapixel_pipeline' if os.path.exists('/content/gigapixel_pipeline') \
         else '/content/gigapixel_cpu'
label  = 'GPU' if 'pipeline' in binary else 'CPU'
print(f'Using {label} binary: {binary}')

os.chdir('/content')

for op_name, op_input in [('Grayscale', '1\n')
 , ('Rotate 90 CW', '2\n')
]:
    print(f'\n=== {label} pipeline ({op_name}) ===')
    t0 = time.perf_counter()
    r  = subprocess.run([binary], input=op_input, text=True, capture_output=True)
    dt = time.perf_counter() - t0
    print(r.stdout[-2000:])
    if r.stderr: print('STDERR:', r.stderr[-500:])
    mp_s = (4096 * 4096 / 1e6) / dt
    print(f'Elapsed: {dt:.2f}s  |  Throughput: {mp_s:.1f} MP/s')

Using CPU binary: /content/gigapixel_cpu

=== CPU pipeline (Grayscale) ===
Popped tile (3072,3072). Requesting VRAM...
[AsyncGPUWorker] CPU stub: falling back for tile (3072,3072)
[Worker 0] Finished tile (3072,3072). Submitting to Writer...
[Writer] Wrote tile at (3072,3072)  [79]
[Worker 0] Popped tile (3584,3072). Requesting VRAM...
[AsyncGPUWorker] CPU stub: falling back for tile (3584,3072)
[Worker 0] Finished tile (3584,3072). Submitting to Writer...
[Reader] Tile 81/84 | Read (4096,3072) -> Write (4096,3072)
[Writer] Wrote tile at (3584,3072)  [80]
[Reader] Tile 82/84 | Read (4608,3072) -> Write (4608,3072)
[Worker 0] Popped tile (4096,3072). Requesting VRAM...
[AsyncGPUWorker] CPU stub: falling back for tile (4096,3072)
[Worker 0] Finished tile (4096,3072). Submitting to Writer...
[Writer] Wrote tile at (4096,3072)  [81]
[Reader] Tile 83/84 | Read (5120,3072) -> Write (5120,3072)
[Worker 0] Popped tile (4608,3072). Requesting VRAM...
[AsyncGPUWorker] CPU stub: falling back for 